In [16]:
import json
import pandas as pd

In [17]:
data = []

with open("../data/candidates.jsonl","r") as f:
    for line in f:
        data.append(json.loads(line))

In [18]:
def create_candidate_text(candidate):

    profile = candidate["profile"]

    text = ""

    # Profile
    text += f"Current title: {profile['current_title']}\n"
    text += f"Summary: {profile['summary']}\n"

    # Experience
    text += "\nExperience:\n"

    for job in candidate["career_history"]:
        text += (
            f"{job['title']} at {job['company']}. "
            f"{job['description']}\n"
        )

    # Skills
    skill_names = [
        skill["name"]
        for skill in candidate["skills"]
    ]

    text += "\nSkills:\n"
    text += ", ".join(skill_names)

    # Education
    for edu in candidate["education"]:
        text += (
            f"\nEducation: "
            f"{edu['degree']} "
            f"{edu['field_of_study']}"
        )

    return text



    

In [19]:
from datetime import datetime

def is_honeypot(candidate):
    profile = candidate.get("profile", {})
    signals = candidate.get("redrob_signals", {})
    
    # 1. Negative experience trap
    if profile.get("years_of_experience", 0) < 0:
        return True
        
    # 2. Temporal paradox trap (signed up after their last active date)
    try:
        signup = datetime.strptime(signals.get("signup_date", "1970-01-01"), "%Y-%m-%d")
        last_active = datetime.strptime(signals.get("last_active_date", "1970-01-01"), "%Y-%m-%d")
        if signup > last_active:
            return True
    except (ValueError, TypeError):
        pass # Skip if dates are missing or corrupted
        
    return False

# UPDATE YOUR MAIN LOOP LIKE THIS:


In [25]:
clean_data = []

for candidate in data:
    if is_honeypot(candidate):
        continue # Drop the trap candidate!
        
    # Proceed with your existing text extraction
    text = create_candidate_text(candidate)
    
    # Using 'candidate_text' to keep it consistent with your dataframe
    clean_data.append({
        "candidate_id": candidate["candidate_id"], 
        "candidate_text": text
    })

# Recreate your DataFrame without the honeypots
candidate_df = pd.DataFrame(clean_data)

print(f"Original candidates: {len(data)}")
print(f"Clean candidates after filter: {len(candidate_df)}")

candidate_df.head()

Original candidates: 100000
Clean candidates after filter: 92504


,candidate_id,candidate_text
0,CAND_0000001,Current title: Backend Engineer\nSummary: Soft...
1,CAND_0000002,Current title: Operations Manager\nSummary: Pr...
2,CAND_0000003,Current title: Customer Support\nSummary: Prof...
3,CAND_0000004,Current title: Marketing Manager\nSummary: Pro...
4,CAND_0000005,Current title: Accountant\nSummary: Profession...


In [26]:
candidate_texts = [
    create_candidate_text(c)
    for c in data
]

In [27]:
print(candidate_texts[0])

Current title: Backend Engineer
Summary: Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.

Experience:
Backend Engineer at Mindtree. Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the dat

In [28]:
candidate_df = pd.DataFrame({
    "candidate_id":[c["candidate_id"] for c in data],
    "candidate_text":candidate_texts
})

In [29]:
candidate_df.head()

,candidate_id,candidate_text
0,CAND_0000001,Current title: Backend Engineer\nSummary: Soft...
1,CAND_0000002,Current title: Operations Manager\nSummary: Pr...
2,CAND_0000003,Current title: Customer Support\nSummary: Prof...
3,CAND_0000004,Current title: Marketing Manager\nSummary: Pro...
4,CAND_0000005,Current title: Accountant\nSummary: Profession...


In [30]:
candidate_df.to_csv(
    "../data/candidate_texts.csv",
    index=False
)

: 